هدفه الدفتر :

13.26 GB CSV   
      ↓   
Streaming Read   
      ↓   
فحص كل سجل دون تحميل الملف كاملًا   
      ↓   
إحصائيات Data Quality   
      ↓   
تحديد الأخطاء الموجودة فعليًا   

# 02 - Data Profiling

## Midterm Big Data Pipeline

This notebook profiles the large raw CSV dataset using streaming-based processing.

### Objectives

- Count the total number of records.
- Identify missing and invalid values.
- Measure the frequency of data quality issues.
- Understand the actual quality distribution of the dataset.
- Use the profiling results to design deterministic cleaning and quarantine rules.

> The dataset is approximately 13 GB.
> The complete file must not be loaded into memory.

In [1]:
from pathlib import Path
import csv
import json
import re
from collections import Counter

In [2]:
PROJECT_ROOT = Path.cwd().parent

DATA_FILE = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "orders_huge_mixed_quality.csv"
)

DATA_FILE

WindowsPath('g:/Semestr_7/Amaly/Big Data/lec 5/H.W.BigData/data/raw/orders_huge_mixed_quality.csv')

In [3]:
if not DATA_FILE.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_FILE}")

print(f"Profiling dataset: {DATA_FILE}")

Profiling dataset: g:\Semestr_7\Amaly\Big Data\lec 5\H.W.BigData\data\raw\orders_huge_mixed_quality.csv


### عدّ جميع السجلات Streaming بدون تحميل الملف للذاكرة.

In [4]:
import time

start_time = time.perf_counter()

row_count = 0

with open(DATA_FILE, "r", encoding="utf-8-sig", newline="") as f:
    reader = csv.reader(f)
    
    # تخطي Header
    next(reader)
    
    for _ in reader:
        row_count += 1

elapsed = time.perf_counter() - start_time

print(f"Total rows: {row_count:,}")
print(f"Elapsed time: {elapsed:.2f} seconds")
print(f"Rows/sec: {row_count / elapsed:,.2f}")

Total rows: 30,000,000
Elapsed time: 157.32 seconds
Rows/sec: 190,690.05


### Profiling للقيم المفقودة

سنفحص 30 مليون سجل مرة أخرى، لكن هذه المرة سنحسب القيم الفارغة لكل عمود.

لأننا نحتاج أن نعرف الأخطاء الموجودة فعلًا في الملف قبل أن نصمم quality_rules.py

In [5]:
from collections import Counter
import time

start_time = time.perf_counter()

missing_counts = Counter()
rows_profiled = 0

with open(DATA_FILE, "r", encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)

    for row in reader:
        rows_profiled += 1

        for field, value in row.items():
            if value is None or value.strip() == "":
                missing_counts[field] += 1

elapsed = time.perf_counter() - start_time

print(f"Rows profiled: {rows_profiled:,}")
print(f"Elapsed time: {elapsed:.2f} seconds")
print("\nMissing values:")

for field, count in missing_counts.items():
    print(f"{field}: {count:,}")

Rows profiled: 30,000,000
Elapsed time: 251.59 seconds

Missing values:
customer_id: 419,474
order_id: 209,392


### فحص القيم غير الطبيعية.

<div dir='rtl'>

فحص items_json تحديدًا، لأنه من أهم الحقول في  البيانات

CORRUPTED_ITEMS_JSON   
EMPTY_ITEMS

سنطبق Streaming على الـ30 مليون سجل، بدون تحميل الملف للذاكرة.

#### لماذا نعمل Profiling

لأن ملف المتطلبات وضع لنا  8 قواعد تنظيف على الأقل،

 وأعطانا أمثلة للأخطاءلذالك من الأفضل أن نعرف أولًا

ما الأخطاء الموجودة فعلًا في الـ30 مليون سجل؟

ثم نصمم القواعد بناءً عليها.

In [6]:
import json
import time

start_time = time.perf_counter()

invalid_items_json = 0
empty_items = 0
valid_items_json = 0

with open(DATA_FILE, "r", encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)

    for row in reader:
        value = row["items_json"]

        if value is None or value.strip() == "":
            empty_items += 1
            continue

        try:
            items = json.loads(value)

            if not isinstance(items, list) or len(items) == 0:
                empty_items += 1
            else:
                valid_items_json += 1

        except (json.JSONDecodeError, TypeError):
            invalid_items_json += 1

elapsed = time.perf_counter() - start_time

print(f"Valid items_json:   {valid_items_json:,}")
print(f"Invalid items_json: {invalid_items_json:,}")
print(f"Empty items:        {empty_items:,}")
print(f"Elapsed time:       {elapsed:.2f} seconds")

Valid items_json:   29,370,160
Invalid items_json: 419,906
Empty items:        209,934
Elapsed time:       335.79 seconds


### الآن أصبح لدينا 4 حالات مهمة

order_id مفقود   
→ MISSING_ORDER_ID  
→ Quarantine  
  
customer_id مفقود  
→ MISSING_CUSTOMER_ID  
→ Quarantine  

items_json تالف  
→ CORRUPTED_ITEMS_JSON  
→ Quarantine  

items فارغة  
→ EMPTY_ITEMS  
→ Quarantine  

<div dir='rtl'>

### فحص التواريخ والقيم الرقمية، لأن الدكتور طلب التعامل مع:

التاريخ
الأرقام العربية
السعر
القيم السالبة
الإجمالي
العملة

سنبدأ بـ order_date.

الهدف هو معرفة:

كم تاريخًا صالحًا؟
كم تاريخًا بتنسيق مختلف؟
كم تاريخًا غير قابل للتحليل؟
هل توجد تواريخ مستحيلة؟

In [7]:
from datetime import datetime
import time

start_time = time.perf_counter()

valid_dates = 0
invalid_dates = 0
empty_dates = 0

date_formats = {
    "%Y-%m-%dT%H:%M:%S": 0,
    "%d/%m/%Y": 0,
    "%d-%m-%Y": 0,
}

with open(DATA_FILE, "r", encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)

    for row in reader:
        value = row["order_date"]

        if value is None or value.strip() == "":
            empty_dates += 1
            continue

        value = value.strip()
        parsed = False

        for fmt in date_formats:
            try:
                datetime.strptime(value, fmt)
                date_formats[fmt] += 1
                valid_dates += 1
                parsed = True
                break
            except ValueError:
                pass

        if not parsed:
            invalid_dates += 1

elapsed = time.perf_counter() - start_time

print(f"Valid dates:   {valid_dates:,}")
print(f"Invalid dates: {invalid_dates:,}")
print(f"Empty dates:   {empty_dates:,}")

print("\nDate formats:")
for fmt, count in date_formats.items():
    print(f"{fmt}: {count:,}")

print(f"\nElapsed time: {elapsed:.2f} seconds")

Valid dates:   29,121,325
Invalid dates: 878,675
Empty dates:   0

Date formats:
%Y-%m-%dT%H:%M:%S: 29,121,325
%d/%m/%Y: 0
%d-%m-%Y: 0

Elapsed time: 435.57 seconds


<div dir='rtl'>

### فحص القيم الرقمية

سأبدأ بـ delivery_cost و payment_amount و total_amount.

نريد معرفة:  

القيم الرقمية الطبيعية.  
الأرقام العربية مثل ٧٠٦٠٠٠٫٠.  
القيم التي تحتوي عملة أو فواصل.  
القيم السالبة.  
القيم غير القابلة للتحويل.  
القيم الفارغة.  

 سنفحصها بدون تعديل البيانات.  

In [8]:
import re
import time

NUMERIC_FIELDS = [
    "delivery_cost",
    "payment_amount",
    "total_amount",
]

ARABIC_DIGITS = str.maketrans(
    "٠١٢٣٤٥٦٧٨٩",
    "0123456789"
)

PERSIAN_DIGITS = str.maketrans(
    "۰۱۲۳۴۵۶۷۸۹",
    "0123456789"
)


def normalize_numeric_text(value):
    """Normalize Arabic/Persian digits and decimal separators."""
    value = value.strip()
    value = value.translate(ARABIC_DIGITS)
    value = value.translate(PERSIAN_DIGITS)
    value = value.replace("٫", ".")
    value = value.replace("٬", ",")
    return value


start_time = time.perf_counter()

stats = {
    field: {
        "valid_numeric": 0,
        "arabic_digits": 0,
        "comma_values": 0,
        "negative_values": 0,
        "invalid_values": 0,
        "empty_values": 0,
    }
    for field in NUMERIC_FIELDS
}

with open(DATA_FILE, "r", encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)

    for row in reader:
        for field in NUMERIC_FIELDS:
            raw_value = row[field]

            if raw_value is None or raw_value.strip() == "":
                stats[field]["empty_values"] += 1
                continue

            value = raw_value.strip()

            normalized = normalize_numeric_text(value)

            if normalized != value:
                stats[field]["arabic_digits"] += 1

            if "," in normalized:
                stats[field]["comma_values"] += 1

            try:
                number = float(normalized.replace(",", ""))

                stats[field]["valid_numeric"] += 1

                if number < 0:
                    stats[field]["negative_values"] += 1

            except ValueError:
                stats[field]["invalid_values"] += 1


elapsed = time.perf_counter() - start_time

for field, field_stats in stats.items():
    print(f"\n=== {field} ===")

    for metric, count in field_stats.items():
        print(f"{metric}: {count:,}")

print(f"\nElapsed time: {elapsed:.2f} seconds")


=== delivery_cost ===
valid_numeric: 29,666,431
arabic_digits: 334,430
comma_values: 0
negative_values: 0
invalid_values: 333,569
empty_values: 0

=== payment_amount ===
valid_numeric: 29,666,112
arabic_digits: 334,540
comma_values: 0
negative_values: 209,114
invalid_values: 333,888
empty_values: 0

=== total_amount ===
valid_numeric: 29,790,568
arabic_digits: 334,299
comma_values: 333,494
negative_values: 0
invalid_values: 209,432
empty_values: 0

Elapsed time: 730.11 seconds


رؤية أمثلة الأخطاء

الأعداد وحدها لا تكفي.

مثلًا عندنا:

payment_amount  
209,114 قيمة سالبة

نريد أن نعرف:  

هل هي:  
- -5000  
- -5,000 ريال  
- -٥٠٠٠  
- نص آخر؟  

وكذلك invalid_values.

In [9]:
import json

SAMPLE_LIMIT = 10

samples = {
    "delivery_cost_invalid": [],
    "payment_amount_invalid": [],
    "payment_amount_negative": [],
    "total_amount_invalid": [],
    "total_amount_comma": [],
}

with open(DATA_FILE, "r", encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)

    for row in reader:

        # delivery_cost
        value = row["delivery_cost"].strip()

        if value:
            try:
                float(normalize_numeric_text(value).replace(",", ""))
            except ValueError:
                if len(samples["delivery_cost_invalid"]) < SAMPLE_LIMIT:
                    samples["delivery_cost_invalid"].append(value)

        # payment_amount
        value = row["payment_amount"].strip()

        if value:
            normalized = normalize_numeric_text(value)

            try:
                number = float(normalized.replace(",", ""))

                if number < 0 and len(samples["payment_amount_negative"]) < SAMPLE_LIMIT:
                    samples["payment_amount_negative"].append(value)

            except ValueError:
                if len(samples["payment_amount_invalid"]) < SAMPLE_LIMIT:
                    samples["payment_amount_invalid"].append(value)

        # total_amount
        value = row["total_amount"].strip()

        if value:
            normalized = normalize_numeric_text(value)

            try:
                float(normalized.replace(",", ""))

                if "," in normalized and len(samples["total_amount_comma"]) < SAMPLE_LIMIT:
                    samples["total_amount_comma"].append(value)

            except ValueError:
                if len(samples["total_amount_invalid"]) < SAMPLE_LIMIT:
                    samples["total_amount_invalid"].append(value)

for category, values in samples.items():
    print(f"\n=== {category} ===")
    for value in values:
        print(repr(value))


=== delivery_cost_invalid ===
'ألفان'
'ألفان'
'ألفان'
'خمسة آلاف'
'خمسة آلاف'
'ألفان'
'خمسة آلاف'
'خمسة آلاف'
'ألفان'
'ألفان'

=== payment_amount_invalid ===
'54000.00 ريال'
'342000.00 ريال'
'421500.00 ريال'
'78500.00 ريال'
'413500.00 ريال'
'86000.00 ريال'
'384000.00 ريال'
'955000.00 ريال'
'446000.00 ريال'
'755500.00 ريال'

=== payment_amount_negative ===
'-21500.0'
'-31000.0'
'-79000.0'
'-413000.0'
'-249000.0'
'-378500.0'
'-33500.0'
'-351500.0'
'-12000.0'
'-564000.0'

=== total_amount_invalid ===
'???'
'???'
'???'
'???'
'???'
'???'
'???'
'???'
'???'
'???'

=== total_amount_comma ===
'135,000.00'
'32,500.00'
'37,500.00'
'152,500.00'
'30,000.00'
'437,000.00'
'59,000.00'
'78,500.00'
'455,500.00'
'65,500.00'


### Profiling للحقول النصية

فحص القيم غير الطبيعية في:  

status  
payment_status  
currency  
delivery_type  
payment_method  

وأيضًا customer_email وcustomer_phone.  

In [10]:
from collections import Counter

TEXT_FIELDS = [
    "status",
    "payment_status",
    "currency",
    "delivery_type",
    "payment_method"
]

counters = {field: Counter() for field in TEXT_FIELDS}

with open(DATA_FILE, "r", encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)

    for row in reader:
        for field in TEXT_FIELDS:
            value = row[field].strip()
            counters[field][value] += 1

for field in TEXT_FIELDS:
    print(f"\n=== {field} ===")

    for value, count in counters[field].most_common():
        print(f"{repr(value)} -> {count:,}")


=== status ===
'قيد الشحن' -> 4,966,741
'قيد الانتظار' -> 4,965,803
'ملغي' -> 4,965,715
'مرتجع' -> 4,964,419
'مؤكد' -> 4,963,804
'تم التسليم' -> 4,963,324
'حالة غير معروفة تمامًا' -> 210,194

=== payment_status ===
'بانتظار الدفع' -> 19,775,790
'تم الدفع' -> 9,890,424
'قيد الدفع' -> 222,692
'مدفوع' -> 111,094

=== currency ===
'YER' -> 29,454,843
'ريال يمني' -> 334,967
'عملة غير معروفة' -> 210,190

=== delivery_type ===
'سريع' -> 15,005,141
'عادي' -> 14,994,859

=== payment_method ===
'محفظة إلكترونية' -> 10,002,523
'نقدًا عند التسليم' -> 9,998,775
'بطاقة' -> 9,998,702


### فحص للـEmail والـPhone

In [11]:
from collections import Counter

EMAIL_SAMPLE_LIMIT = 10
PHONE_SAMPLE_LIMIT = 10

email_samples = []
phone_samples = []

email_counter = Counter()
phone_counter = Counter()

with open(DATA_FILE, "r", encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)

    for row in reader:
        email = row["customer_email"].strip()
        phone = row["customer_phone"].strip()

        email_counter[email] += 1
        phone_counter[phone] += 1

        # أمثلة البريد غير الطبيعي
        if (
            not email
            or "@" not in email
            or email.count("@") != 1
            or ".." in email
        ):
            if len(email_samples) < EMAIL_SAMPLE_LIMIT:
                email_samples.append(email)

        # أمثلة الهاتف غير الطبيعي
        digits_only = "".join(ch for ch in phone if ch.isdigit())

        if (
            not phone
            or len(digits_only) < 9
            or phone != digits_only
        ):
            if len(phone_samples) < PHONE_SAMPLE_LIMIT:
                phone_samples.append(phone)


print("=== Email Samples ===")
for value in email_samples:
    print(repr(value))

print("\n=== Phone Samples ===")
for value in phone_samples:
    print(repr(value))

=== Email Samples ===
'@@'
'user819896@@example.com'
'user100791@@example.com'
'user306743@@example.com'
'user393888@@example.com'
'user714581@example..com'
'user801356@@example.com'
'user482220@example..com'
'user-without-domain'
'user356028@example..com'

=== Phone Samples ===
'77 557 8449'
'+967 777559764'
'12345'
'71 759 2278'
'12345'
'+967 777548975'
'12345'
'12345'
'73 541 9770'
'71 792 0145'


### order_id فحص  تكرار الـ



In [12]:
from collections import Counter

order_id_counter = Counter()

with open(DATA_FILE, "r", encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)

    for row in reader:
        order_id = row["order_id"].strip()
        order_id_counter[order_id] += 1


total_rows = sum(order_id_counter.values())
unique_order_ids = len(order_id_counter)
duplicate_rows = sum(
    count - 1
    for count in order_id_counter.values()
    if count > 1
)

duplicate_order_ids = sum(
    1
    for count in order_id_counter.values()
    if count > 1
)

missing_order_ids = order_id_counter.get("", 0)

print("=== order_id Profiling ===")
print(f"Total rows: {total_rows:,}")
print(f"Unique order_ids: {unique_order_ids:,}")
print(f"Duplicate order_ids: {duplicate_order_ids:,}")
print(f"Duplicate rows: {duplicate_rows:,}")
print(f"Missing order_ids: {missing_order_ids:,}")

print("\nSample duplicated order_ids:")

shown = 0

for order_id, count in order_id_counter.items():
    if count > 1 and order_id:
        print(f"{repr(order_id)} -> {count}")
        shown += 1

        if shown >= 10:
            break

=== order_id Profiling ===
Total rows: 30,000,000
Unique order_ids: 29,580,714
Duplicate order_ids: 207,690
Duplicate rows: 419,286
Missing order_ids: 209,392

Sample duplicated order_ids:
'طلب-100013' -> 3
'طلب-100019' -> 2
'طلب-100024' -> 2
'طلب-100025' -> 2
'طلب-100027' -> 4
'طلب-100047' -> 2
'طلب-100076' -> 2
'طلب-100100' -> 2
'طلب-100110' -> 2
'طلب-100120' -> 2
